In [54]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import time

options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()),
                          options=options)


idx = pd.IndexSlice

In [55]:
# title : //ol[1]/li[1]/div/div[2]/div[2]/a
# rank : //ol[1]/li[2]/div/div[2]/div[2]/div[1]/div[1]/div/div/span
# 저자/출판사/출판일: //ol[1]/li[1]/div/div[2]/div[2]/div[2]
# price : //ol[1]/li[1]/div/div[2]/div[2]/div[3]/div/span[2]/span[1]
# 할인율: //ol[1]/li[1]/div/div[2]/div[2]/div[3]/div/span[1]
# 리뷰: //ol[1]/li[1]/div/div[2]/div[2]/div[4]/div/div/div/div/span[1]


In [56]:
def parse_item(item):
    # 제목
    title_tag = item.select_one("a.prod_link.line-clamp-2.font-medium")
    title = title_tag.get_text(strip=True) if title_tag else None

    # 순위
    rank_tag = item.select_one("div.flex.flex-col.gap-3 span.font-bold")
    rank = rank_tag.get_text(strip=True) if rank_tag else None

    # 저자 / 출판사 / 출판일
    info_tag = item.select_one("div.line-clamp-2.fz-14")
    author = publisher = pub_date = None
    if info_tag:
        info_text = info_tag.get_text(" ", strip=True)
        parts = [x.strip() for x in info_text.split("·")]
        if len(parts) >= 3:
            author, publisher, pub_date = parts[:3]

    # 평점
    score_tag = item.select_one("span.font-bold.text-black")
    score = score_tag.get_text(strip=True) if score_tag else None

    # 할인율
    discount_tag = item.select_one("span.font-bold.inline-block.text-green-800")
    discount_rate = discount_tag.get_text(strip=True) if discount_tag else None

    # 판매가격
    price_tag = item.select_one("span.inline-block.align-top.fz-16 span.font-bold")
    price = price_tag.get_text(strip=True) if price_tag else None

    return {
        "rank": rank,
        "title": title,
        "author": author,
        "publisher": publisher,
        "pub_date": pub_date,
        "score": score,
        "discount": discount_rate,
        "price": price,
        "site": '교보문고'
    }


In [57]:
def kyobo_month_bestseller(ym):
    url = f"https://store.kyobobook.co.kr/bestseller/total/monthly?ym={ym}&page=1&per=50"

    driver.get(url)
    time.sleep(2)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # ★ 여기 다시 mt-9로!
    items = soup.select("li.mt-9")

    books = []
    for it in items:
        data = parse_item(it)
        data["ym"] = ym
        books.append(data)

    return pd.DataFrame(books)


In [58]:
def multi_month_bestseller(start_ym="202412", end_ym="202512"):
    start_year, start_month = int(start_ym[:4]), int(start_ym[4:])
    end_year, end_month = int(end_ym[:4]), int(end_ym[4:])

    all_data = pd.DataFrame()
    year, month = start_year, start_month

    print("=== 교보문고 월간 베스트 수집 시작 ===")

    while True:
        ym = f"{year}{month:02d}"
        print(f"📌 수집 중: {ym}")

        try:
            df_month = kyobo_month_bestseller(ym)
            all_data = pd.concat([all_data, df_month], ignore_index=True)
        except Exception as e:
            print(f"❌ 오류 발생 ({ym}): {e}")

        if year == end_year and month == end_month:
            break

        month += 1
        if month > 12:
            month = 1
            year += 1

    print("✅ 전체 수집 완료")
    return all_data


In [59]:
driver.get("https://store.kyobobook.co.kr/bestseller/total/monthly?ym=202412&page=1&per=50")
time.sleep(2)

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

items = soup.select("li.mt-9")
len(items)


50

In [63]:
def save_to_csv(df, filename):
    df.to_csv(filename, encoding="utf-8-sig", index=False)
    print(f"📁 CSV 저장 완료: {filename}")


In [61]:
print(items[0].prettify())


<li class="mt-9 flex min-h-[170px] flex-col justify-center pt-9 border-t border-gray-300 first:mt-0 first:border-t-0 first:pt-0">
 <div class="flex">
  <div class="mr-2 w-[22px] text-center">
   <label class="relative min-w-[22px] min-h-[22px] cursor-pointer mr-2.5 h-[22px]">
    <input class="absolute h-1 w-1 opacity-0" type="checkbox"/>
    <div class="absolute w-5 h-5 rounded flex justify-center items-center border transition-all duration-200 ease-out box-border bg-white border-gray-500">
     <img alt="check" class="rounded-full" data-nimg="1" decoding="async" height="7" loading="lazy" src="https://contents.kyobobook.co.kr/resources/fo/images/common/ink/ico_checkbox@2x.png" style="color: transparent;" width="10"/>
    </div>
    <div class="text-text font-text-l relative cursor-pointer pl-6">
     <span class="hidden">
      상품선택
     </span>
    </div>
   </label>
  </div>
  <div class="flex items-top justify-start mr-[35px] w-full">
   <div class="relative flex-shrink-0 overflow-